In [2]:
import pandas as pd
from sodapy import Socrata

client = Socrata("data.cityofchicago.org", None, timeout=60)

socio = client.get("kn9c-c2s2", limit=100)
socio_df = pd.DataFrame.from_records(socio)
print(socio_df.columns.tolist())
print(f"{len(socio_df)} rows")
socio_df.head()

['ca', 'community_area_name', 'percent_of_housing_crowded', 'percent_households_below_poverty', 'percent_aged_16_unemployed', 'percent_aged_25_without_high_school_diploma', 'percent_aged_under_18_or_over_64', 'per_capita_income_', 'hardship_index']
78 rows


,ca,community_area_name,percent_of_housing_crowded,percent_households_below_poverty,percent_aged_16_unemployed,percent_aged_25_without_high_school_diploma,percent_aged_under_18_or_over_64,per_capita_income_,hardship_index
0,1,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39
1,2,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46
2,3,Uptown,3.8,24,8.9,11.8,22.2,35787,20
3,4,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17
4,5,North Center,0.3,7.5,5.2,4.5,26.2,57123,6


In [3]:
# keep the columns we'll actually use
lookup = socio_df[[
    "ca",
    "community_area_name",
    "percent_households_below_poverty",
    "per_capita_income_",
    "hardship_index"
]].copy()

# rename for clarity
lookup = lookup.rename(columns={
    "ca": "community_area",
    "community_area_name": "area_name",
    "percent_households_below_poverty": "pct_below_poverty",
    "per_capita_income_": "per_capita_income",
    "hardship_index": "hardship_index"
})

# cast types
lookup["community_area"] = pd.to_numeric(lookup["community_area"], errors="coerce")
for col in ["pct_below_poverty", "per_capita_income", "hardship_index"]:
    lookup[col] = pd.to_numeric(lookup[col], errors="coerce")

# drop the citywide total row (it has no community_area number)
lookup = lookup.dropna(subset=["community_area"])
lookup["community_area"] = lookup["community_area"].astype(int)

print(f"{len(lookup)} community areas")  # should be 77
print(lookup.dtypes)
lookup.head()

77 community areas
community_area         int64
area_name                str
pct_below_poverty    float64
per_capita_income      int64
hardship_index       float64
dtype: object


,community_area,area_name,pct_below_poverty,per_capita_income,hardship_index
0,1,Rogers Park,23.6,23939,39.0
1,2,West Ridge,17.2,23040,46.0
2,3,Uptown,24.0,35787,20.0
3,4,Lincoln Square,10.9,37524,17.0
4,5,North Center,7.5,57123,6.0


In [4]:
import os
lookup.to_parquet("data/community_lookup.parquet", index=False)
print(f"Saved. {os.listdir('data')}")

Saved. ['crimes_raw.parquet', 'community_lookup.parquet', 'crimes_clean.parquet']
